## Imports og tilkobling

In [1]:
import sys
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

sys.path.append(os.path.abspath("../../../"))
from src.feature_engineering.forbruksdata import create_consumption_features

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "rawdata"

## Hente CSV fra blob

In [2]:
blob_name = "AntallMålepunktNorgespris.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

## Feature engineering

In [6]:
from src.feature_engineering.norgespris import clean_norgespris_df, split_by_station

df = clean_norgespris_df(df)

station_dfs = split_by_station(df)

df.head()

,transformer_station,timestamp,count_total
0,TIMENES TS,2025-10-01 00:00:00,1946
1,TIMENES TS,2025-10-02 00:00:00,1990
2,TIMENES TS,2025-10-03 00:00:00,2022
3,TIMENES TS,2025-10-04 00:00:00,2040
4,TIMENES TS,2025-10-05 00:00:00,2055


## Lagre i Azure Blob

In [7]:
output_container = "processeddata"

for station_name, station_df in station_dfs.items():
    
    output_blob_name = f"{station_name}_norgespris.csv"
    
    csv_buffer = io.StringIO()
    station_df.to_csv(csv_buffer, index=False)

    blob_client = blob_service_client.get_blob_client(
        container=output_container,
        blob=output_blob_name
    )

    blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)

    print(f"Lastet opp: {output_blob_name}")

Lastet opp: breive_norgespris.csv
Lastet opp: frikstad_norgespris.csv
Lastet opp: hartevatn_norgespris.csv
Lastet opp: timenes_ts_norgespris.csv
